# Camada Gold: Exportação para Power BI

Exportação idempotente das tabelas consolidadas da camada Gold para arquivos CSV limpos no volume da Landing Zone, viabilizando o download direto e a importação posterior no Power BI Desktop.

In [0]:
%run ./00_Config

In [0]:
# Importação das bibliotecas necessárias
import os
import pandas as pd

# Caminho de destino para exportação no volume da Landing Zone
export_path = f"{landing_path}/export_power_bi"

# Assegura que o diretório de destino existe no ambiente
dbutils.fs.mkdirs(export_path)

# Mapeamento das tabelas Gold a serem exportadas para arquivos CSV limpos
tabelas_para_exportar = {
    "fato_chamados.csv": f"{gold_schema}.fato_chamados",
    "agg_visao_executiva_mensal.csv": f"{gold_schema}.agg_visao_executiva_mensal",
    "dim_canais.csv": f"{gold_schema}.dim_canais",
    "dim_motivos.csv": f"{gold_schema}.dim_motivos",
    "dim_atendentes.csv": f"{gold_schema}.dim_atendentes",
    "dim_clientes.csv": f"{gold_schema}.dim_clientes",
    "dim_data.csv": f"{gold_schema}.dim_data"
}

# Execução da exportação convertendo para Pandas para gerar um arquivo CSV único por tabela
for nome_arquivo, tabela_delta in tabelas_para_exportar.items():
    # Coleta os dados consolidados da tabela Delta para um DataFrame do Pandas
    df_pandas = spark.table(tabela_delta).toPandas()
    
    # Caminho completo do arquivo de saída
    caminho_arquivo = f"{export_path}/{nome_arquivo}"
    
    # Salva o arquivo CSV limpo com cabeçalho e codificação UTF-8
    df_pandas.to_csv(caminho_arquivo, index=False, encoding="utf-8")
    
    print(f"[OK] Tabela {tabela_delta} exportada com sucesso para {caminho_arquivo}")